# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a dataset defined by a Croissant schema using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset elements are referenced by their unique `@id` fields for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

> **Note:** You can use this notebook as a base for working with any Croissant schema-powered dataset.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata & Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
We will list available record sets and their details using their `@id`.

> **Tip:** All entities (record sets, fields, columns, etc.) should be referenced by their `@id`.

**Record sets in this dataset:**

In [ ]:
# List record sets with their @id and fields
record_sets = []

print("Record Sets:")
for rs in metadata.record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name']}")
    record_sets.append(rs['@id'])
    if rs.get('fields'):
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - @id: {field['@id']}, name: {field['name']}, dataType: {field.get('dataType','?')}")
    print()

# We'll use the first record set for exploration
if record_sets:
    print(f"Selected record set: {record_sets[0]}")
else:
    print('No record sets found.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id` values from the overview above.

We'll load all record sets into dataframes, keyed by their `@id`.

In [ ]:
# Load each record set into a pandas DataFrame using its @id
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  -> Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  -> No records in this record set.")

# Pick first record set with data for further analysis
main_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break
if main_record_set_id:
    print(f"Selected for analysis: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing, and grouping, using fields by their `@id`.

First, list all numeric fields in the selected record set.

In [ ]:
# Identify numeric fields (as per Croissant schema in main_record_set_id)
import numpy as np

numeric_field_ids = []
group_field_id = None

for rs in metadata.record_sets:
    if rs['@id'] == main_record_set_id:
        # Try to detect numeric fields
        for field in rs.get('fields', []):
            dt = field.get('dataType', '')
            # Some schemas use cr:Float/cr:Integer etc
            if 'Float' in dt or 'Integer' in dt or dt in ['float', 'number', 'integer']:
                numeric_field_ids.append(field['@id'])
            # Pick the first non-numeric field as possible group field
            if not group_field_id and not (('Float' in dt) or ('Integer' in dt) or dt in ['float', 'number', 'integer']):
                group_field_id = field['@id']

print(f"Numeric fields (@id): {numeric_field_ids}")
if group_field_id:
    print(f"Candidate group field (@id): {group_field_id}")

In [ ]:
# We'll proceed with the first available numeric field and the candidate group field
if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
else:
    print("No numeric fields available.")
    numeric_field = None

group_field = group_field_id

df = dataframes[main_record_set_id]

if numeric_field and numeric_field in df.columns:
    print(f"Working with numeric field '@id': {numeric_field}")

    # Try to ensure numeric; coerce errors to NaN
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if pd.notna(df[numeric_field].mean()) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() or 1)
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No available numeric field in the dataframe for EDA.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and (optionally) group means by the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If grouped statistics were computed above
    if group_field and group_field in df.columns:
        grouped_df = df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook we demonstrated: 
* Loading a Croissant-enabled dataset via its schema URL,
* Inspecting available record sets, fields, and their `@id`s,
* Extracting records into DataFrames,
* Performing preliminary EDA and visualization using field references by `@id`.

You can extend this notebook to perform domain-specific analyses, machine learning, or reporting for your own use-cases.

For advanced use, consult the [`mlcroissant` documentation](https://github.com/mlcommons/croissant).